# 🤖 AI Engineering Fundamentals — Lezione 5
## Notebook Gruppo A

**ITS Novitas 4.0 | Giovedì 04/06/2026**

---

### 📋 Istruzioni
1. **File → Salva una copia in Drive** prima di iniziare
2. Lavorate in gruppo — discutete prima di scrivere
3. Alla fine: **File → Scarica → .ipynb** e caricate su GitHub

### 👥 Membri del gruppo

In [1]:
GRUPPO = "A"
MEMBRI = ["Lorenzo Masia", "Carlo Abis", "Alfonso Mammato"]  # ← inserite i vostri nomi
print(f"Gruppo {GRUPPO} — {', '.join(m for m in MEMBRI if m)}")

Gruppo A — Lorenzo Masia, Carlo Abis, Alfonso Mammato


In [2]:
!pip install anthropic requests -q
from google.colab import userdata
import anthropic, os, requests, json

os.environ["ANTHROPIC_API_KEY"] = userdata.get("ANTHROPIC_API_KEY")
client = anthropic.Anthropic()

# Tool loop base
def tool_loop(messaggio, tools, esegui_fn, system=None):
    history = [{"role": "user", "content": messaggio}]
    iterazioni = 0
    while True:
        iterazioni += 1
        if iterazioni > 10:
            return "Errore: loop non terminato"
        params = {"model": "claude-haiku-4-5-20251001",
                  "max_tokens": 1024, "tools": tools, "messages": history}
        if system:
            params["system"] = system
        response = client.messages.create(**params)
        if response.stop_reason == "end_turn":
            return next(b.text for b in response.content if b.type == "text")
        if response.stop_reason == "tool_use":
            history.append({"role": "assistant", "content": response.content})
            results = []
            for b in response.content:
                if b.type == "tool_use":
                    print(f"  🔧 {b.name}({b.input})")
                    r = esegui_fn(b.name, b.input)
                    print(f"  ✅ {str(r)[:100]}")
                    results.append({"type": "tool_result",
                                   "tool_use_id": b.id, "content": str(r)})
            history.append({"role": "user", "content": results})

print("✅ Setup completato!")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 837.5/837.5 kB 10.6 MB/s eta 0:00:00
✅ Setup completato!


---
## 🎯 Tema del Gruppo A: MCP — Model Context Protocol

Esplorate il pattern MCP attraverso un MockMCPServer.
Capite come un server espone tool, come il client li scopre
e come si integrano nel chatbot.

---
### Esercizio 1 — Il MockMCPServer *(guidato)*

Un server MCP reale espone tool via protocollo JSON-RPC.
Il MockMCPServer simula questo pattern in Python puro.
Esplorate come funziona e quali tool espone.

In [3]:
# Esercizio 1 — il MockMCPServer

class MockMCPServer:
    """
    Simula un server MCP per il filesystem locale.
    In un server MCP reale, questi tool sarebbero esposti
    via JSON-RPC su un processo separato.
    """

    def list_tools(self):
        """Restituisce la lista dei tool disponibili."""
        return [
            {
                "name": "leggi_file",
                "description": "Leggi il contenuto di un file di testo locale.",
                "input_schema": {
                    "type": "object",
                    "properties": {
                        "path": {"type": "string", "description": "Percorso del file"}
                    },
                    "required": ["path"]
                }
            },
            {
                "name": "scrivi_file",
                "description": "Scrivi testo in un file. Crea il file se non esiste.",
                "input_schema": {
                    "type": "object",
                    "properties": {
                        "path": {"type": "string", "description": "Percorso del file"},
                        "contenuto": {"type": "string", "description": "Testo da scrivere"}
                    },
                    "required": ["path", "contenuto"]
                }
            },
            {
                "name": "lista_file",
                "description": "Elenca i file in una directory.",
                "input_schema": {
                    "type": "object",
                    "properties": {
                        "directory": {"type": "string", "description": "Percorso della directory"}
                    },
                    "required": ["directory"]
                }
            }
        ]

    def call_tool(self, nome, parametri):
        """Esegue un tool e restituisce il risultato."""
        if nome == "leggi_file":
            path = parametri["path"]
            if os.path.exists(path):
                with open(path, "r", encoding="utf-8") as f:
                    return f.read()[:1000]
            return f"File non trovato: {path}"

        elif nome == "scrivi_file":
            # Minimal footprint: solo nella directory corrente
            path = parametri["path"]
            if "/" in path and not path.startswith("./"):
                return "Errore: scrittura consentita solo nella directory corrente"
            with open(path, "w", encoding="utf-8") as f:
                f.write(parametri["contenuto"])
            return f"File '{path}' scritto con successo"

        elif nome == "lista_file":
            directory = parametri["directory"]
            if os.path.exists(directory):
                files = os.listdir(directory)
                return f"File in '{directory}': {', '.join(files[:20])}"
            return f"Directory non trovata: {directory}"

        return f"Tool '{nome}' non disponibile"

# Inizializza il server
mcp = MockMCPServer()

# TODO: stampate i tool disponibili nel formato
# • nome_tool: descrizione
print("🔌 Tool disponibili dal server MCP:")
for tool in mcp.list_tools():
    # TODO
    print(tool)

# Testate i tool direttamente (senza chatbot)
print()
print("Test diretto:")
print(mcp.call_tool("scrivi_file", {"path": "note_test.txt", "contenuto": "WiData IoT test"}))
print(mcp.call_tool("leggi_file", {"path": "note_test.txt"}))
print(mcp.call_tool("lista_file", {"directory": "."}))

🔌 Tool disponibili dal server MCP:
{'name': 'leggi_file', 'description': 'Leggi il contenuto di un file di testo locale.', 'input_schema': {'type': 'object', 'properties': {'path': {'type': 'string', 'description': 'Percorso del file'}}, 'required': ['path']}}
{'name': 'scrivi_file', 'description': 'Scrivi testo in un file. Crea il file se non esiste.', 'input_schema': {'type': 'object', 'properties': {'path': {'type': 'string', 'description': 'Percorso del file'}, 'contenuto': {'type': 'string', 'description': 'Testo da scrivere'}}, 'required': ['path', 'contenuto']}}
{'name': 'lista_file', 'description': 'Elenca i file in una directory.', 'input_schema': {'type': 'object', 'properties': {'directory': {'type': 'string', 'description': 'Percorso della directory'}}, 'required': ['directory']}}

Test diretto:
File 'note_test.txt' scritto con successo
WiData IoT test
File in '.': .config, note_test.txt, sample_data


---
### Esercizio 2 — Integrare MCP nel chatbot *(guidato)*

I tool del server MCP si integrano esattamente come
i tool custom — passando la lista al loop agente.
Il client non sa se un tool è custom o MCP.

In [5]:
# Esercizio 2 — MCP integrato nel chatbot

# I tool MCP vengono caricati dal server
mcp_tools = mcp.list_tools()

# Tool custom aggiuntivi
tool_calcolatrice = {
    "name": "calcola",
    "description": "Esegui calcoli matematici precisi.",
    "input_schema": {"type": "object",
                     "properties": {"espressione": {"type": "string"}},
                     "required": ["espressione"]}
}

def calcola(esp):
    allowed = set('0123456789+-*/().% ')
    if not all(c in allowed for c in esp): return "Espressione non valida"
    try: return f"{esp} = {eval(esp)}"
    except: return "Errore nel calcolo"

# Router unificato: gestisce sia tool custom che tool MCP
def esegui_tutto(nome, parametri):
    """Router che gestisce tool custom e tool MCP."""
    # Tool custom
    if nome == "calcola":
        return calcola(parametri["espressione"])

    # Tool MCP — smistati al server
    # TODO: se nome è in ["leggi_file", "scrivi_file", "lista_file"]
    # chiamate mcp.call_tool(nome, parametri)
    mcp_toolname = [el_lista["name"] for el_lista in mcp_tools]
    if nome in mcp_toolname:
        return mcp.call_tool(nome, parametri)

    return f"Tool '{nome}' non trovato"

# Tutti i tool insieme
tutti_i_tool = [tool_calcolatrice] + mcp_tools

print(f"Tool disponibili: {[t['name'] for t in tutti_i_tool]}\n")

# Crea un file di test
with open("report_widata.txt", "w", encoding="utf-8") as f:
    f.write("Report sensori WiData - Giugno 2026\n")
    f.write("Sensore XS200-01: temperatura 24.3°C, umidità 65%\n")
    f.write("Sensore XS200-02: temperatura 25.1°C, umidità 62%\n")
    f.write("Sensore XS200-03: ANOMALIA - PM2.5 = 85 µg/m³\n")

# Test: domande che usano sia tool custom che MCP
domande = [
    "Leggi il file report_widata.txt e dimmi se ci sono anomalie",
    "Quanti sensori ci sono nel report e qual è la temperatura media? il report é nella directory corrente",
]

for d in domande:
    print(f"\n❓ {d}")
    # TODO: chiamate tool_loop con tutti_i_tool ed esegui_tutto
    print(tool_loop(d, tutti_i_tool, esegui_tutto, system=None))

Tool disponibili: ['calcola', 'leggi_file', 'scrivi_file', 'lista_file']


❓ Leggi il file report_widata.txt e dimmi se ci sono anomalie
  🔧 leggi_file({'path': 'report_widata.txt'})
  ✅ Report sensori WiData - Giugno 2026
Sensore XS200-01: temperatura 24.3°C, umidità 65%
Sensore XS200-
Ho letto il file. Ecco quello che ho trovato:

**ANOMALIA RILEVATA:**

Nel report dei sensori WiData di giugno 2026, è presente una **anomalia critica** nel **Sensore XS200-03**:

- **PM2.5 = 85 µg/m³** 

Questo valore è molto elevato e rappresenta un'anomalia. Per contesto:
- I livelli di PM2.5 sono considerati **"Cattivi"** sopra i 55.5 µg/m³
- Un valore di 85 µg/m³ indica una **qualità dell'aria scadente** e rappresenta un potenziale rischio per la salute

Gli altri due sensori (XS200-01 e XS200-02) mostrano parametri normali:
- Temperatura: 24.3-25.1°C (buoni valori)
- Umidità: 62-65% (valori normali)

**Consiglio:** Verificare il sensore XS200-03 e investigare la possibile fonte di inquinamento par

---
### Esercizio 3 — Estendere il server MCP *(libero)*

Aggiungete al MockMCPServer un tool `cerca_json`
che legge un file JSON e cerca un valore per chiave.
Utile per interrogare configurazioni o dati strutturati.

In [7]:
# Esercizio 3 — estendere il server MCP

class MockMCPServerV2(MockMCPServer):
    """Versione estesa con tool aggiuntivi."""

    def list_tools(self):
        """Aggiunge cerca_json ai tool esistenti."""
        base = super().list_tools()
        # TODO: aggiungete la definizione del tool cerca_json
        nuovo_tool = {
            "name": "cerca_json",
            "description": "Leggi un file JSON e cerca un valore per chiave, utile per interrogare configurazioni o dati strutturati.",  # ← scrivete voi
            "input_schema": {
                "type": "object",
                "properties": {
                    "path": {"type": "string", "description": "Percorso file JSON"},
                    "chiave": {"type": "string", "description": "Chiave da cercare"}
                },
                "required": ["path", "chiave"]
            }
        }
        return base + [nuovo_tool]

    def call_tool(self, nome, parametri):
        if nome == "cerca_json":
            # TODO: leggete il file JSON e cercate la chiave
            # Gestite il caso in cui il file non esiste
            # Gestite il caso in cui la chiave non esiste
            path = parametri["path"]
            try:
              with open(path, 'r', encoding='utf-8') as file:
                  dati_json = json.load(file)
            except:
              return "File non trovato"

            chiave = parametri["chiave"]
            try:
              return dati_json[chiave]
            except:
              return "Chiave non trovata"

        return super().call_tool(nome, parametri)

# Test
mcp_v2 = MockMCPServerV2()

# Crea un file JSON di test
config = {
    "sensori": 12,
    "gateway": "GW500",
    "soglia_pm25": 75,
    "email_alert": "ops@widata.cloud"
}
with open("config_widata.json", "w") as f:
    json.dump(config, f)

# TODO: testate cerca_json direttamente
print(mcp_v2.call_tool("cerca_json", {"path": "config_widata.json", "chiave": "soglia_pm25"}))
print(mcp_v2.call_tool("cerca_json", {"path": "config_widata.json", "chiave": "chiave_inesistente"}))

# Testate con il chatbot
print()
print(tool_loop(
    "Qual è la soglia di allerta per PM2.5 nella configurazione?",
    mcp_v2.list_tools() + [tool_calcolatrice],
    lambda n, p: calcola(p["espressione"]) if n == "calcola" else mcp_v2.call_tool(n, p)
))

75
Chiave non trovata

Mi occorre il percorso del file di configurazione per cercare la soglia di allerta per PM2.5. Non ho informazioni su dove si trova questo file nel tuo sistema.

Puoi fornirmi:
1. Il percorso completo del file di configurazione (ad esempio `/etc/config.json` o `config/settings.json`)
2. Oppure la directory dove si trova, così posso cercare i file di configurazione disponibili

Una volta che mi dai questa informazione, potrò cercare la soglia di allerta per PM2.5.


---
### Esercizio 4 — MCP vs tool custom: quando usare quale *(libero)*

Costruite un confronto pratico:
stesso task implementato come tool custom e come server MCP.
Quali sono i vantaggi di ciascun approccio?
Quando usereste MCP in un progetto reale come Xplore?

In [8]:
# Esercizio 4 — MCP vs tool custom

# Task: leggere i dati di un sensore da un file

# APPROCCIO A: tool custom
tool_leggi_sensore_custom = {
    "name": "leggi_sensore",
    "description": "Leggi i dati dell'ultimo rilevamento di un sensore WiData.",
    "input_schema": {
        "type": "object",
        "properties": {
            "sensore_id": {"type": "string", "description": "ID del sensore es: XS200-01"}
        },
        "required": ["sensore_id"]
    }
}

def leggi_sensore(sensore_id: str) -> str:
    # Dati simulati
    dati = {
        "XS200-01": {"temp": 24.3, "umidita": 65, "pm25": 12},
        "XS200-02": {"temp": 25.1, "umidita": 62, "pm25": 15},
        "XS200-03": {"temp": 23.8, "umidita": 68, "pm25": 85},
    }
    if sensore_id in dati:
        d = dati[sensore_id]
        return f"{sensore_id}: temp={d['temp']}°C, umidità={d['umidita']}%, PM2.5={d['pm25']}µg/m³"
    return f"Sensore '{sensore_id}' non trovato"

# APPROCCIO B: stesso tool esposto via MockMCPServer
# TODO: create un MockMCPServerSensori che espone leggi_sensore come tool MCP
class MockMCPServerWiData(MockMCPServerV2):
    """Versione estesa con tool aggiuntivi."""

    def list_tools(self):
        """Aggiunge cerca_json ai tool esistenti."""
        base = super().list_tools()
        # Legge sensore
        lettura_sensore_tool = {
            "name": "leggi_sensore",
            "description": "Leggi i dati dell'ultimo rilevamento di un sensore WiData.",
            "input_schema": {
                "type": "object",
                "properties": {
                    "sensore_id": {"type": "string", "description": "ID del sensore es: XS200-01"}
                },
            "required": ["sensore_id"]
    }
        }
        return base + [lettura_sensore_tool]

    def call_tool(self, nome, parametri):
        if nome == "leggi_sensore":
            sensore_id = parametri["sensore_id"]
            # Dati simulati
            dati = {
                "XS200-01": {"temp": 24.3, "umidita": 65, "pm25": 12},
                "XS200-02": {"temp": 25.1, "umidita": 62, "pm25": 15},
                "XS200-03": {"temp": 23.8, "umidita": 68, "pm25": 85},
            }
            if sensore_id in dati:
                d = dati[sensore_id]
                return f"{sensore_id}: temp={d['temp']}°C, umidità={d['umidita']}%, PM2.5={d['pm25']}µg/m³"
            return f"Sensore '{sensore_id}' non trovato"

        return super().call_tool(nome, parametri)


# Confronto:
# 1. Quanto codice in più richiede l'approccio MCP?
# 2. Quale è più facile da aggiornare se cambiano i dati?
# 3. Quale funzionerebbe con un'altra applicazione AI oltre al vostro chatbot?
# 4. Per il progetto Xplore reale, quale approccio scegliereste?




print("Confronto approcci:")
print()
print("Tool custom:")
print(f"  → {leggi_sensore('XS200-03')}")
print()
# TODO: testate anche la versione MCP

# Conclusione del gruppo:
# Tool custom quando: ...
# MCP quando: ...
# Per Xplore useremmo: ...


Confronto approcci:

Tool custom:
  → XS200-03: temp=23.8°C, umidità=68%, PM2.5=85µg/m³



In [10]:
# Test
mcp_widata = MockMCPServerWiData()

# TODO: testate cerca_json direttamente
print(mcp_widata.call_tool("leggi_sensore", sensore_id='XS200-03'))

TypeError: MockMCPServerWiData.call_tool() got an unexpected keyword argument 'sensore_id'

In [ ]:
# Testate con il chatbot
print()
print(tool_loop(
    "Qual è la soglia di allerta per PM2.5 nella configurazione?",
    mcp_widata.list_tools() + [tool_calcolatrice],
    lambda n, p: calcola(p["espressione"]) if n == "calcola" else mcp_widata.call_tool(n, p)
))

---
## 📊 Preparate la presentazione (5 slide)

1. **Cos'è MCP** — il problema che risolve con l'analogia HTTP
2. **Il MockMCPServer** — come espone i tool e come vengono scoperti
3. **MCP integrato nel chatbot** — demo: legge il report anomalie
4. **Tool custom vs MCP** — il vostro confronto con i pro/contro
5. **Per Xplore** — quale architettura scegliereste e perché

---
*ITS Novitas 4.0 — AI Engineering Fundamentals | Marco Uras*